# Capítulo 2. Transformación y análisis descriptivo de los datos de estratificación socioeconómica

## Metodología SEMMA
Fases desarrolladas:
- Modify
- Model

## Proyecto
Análisis descriptivo de la estratificación socioeconómica en los departamentos de Cauca, Antioquia y Cundinamarca

## Objetivo del capítulo
Realizar la limpieza, transformación y análisis descriptivo de los datos socioeconómicos para identificar patrones territoriales y distribución de estratos entre departamentos.

In [ ]:

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import plotly.express as px

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10,6)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 2.1 Limpieza y depuración de los datos

En esta sección se realiza la carga y validación inicial del conjunto de datos para identificar problemas de calidad, inconsistencias y valores faltantes.

In [ ]:

df = pd.read_csv('../data/raw/estratificacion.csv')

df.head()

In [ ]:

print("Dimensiones:", df.shape)

In [ ]:

df.info()

In [ ]:
df.describe(include='all')

## Identificación de valores faltantes

In [ ]:

df.isnull().sum()

In [ ]:
plt.figure(figsize=(10,5))

sns.heatmap(df.isnull(), cbar=False)

plt.title('Mapa de valores faltantes')

plt.show()

In [ ]:
duplicados = df.duplicated().sum()

print("Duplicados:", duplicados)

In [ ]:
df = df.drop_duplicates()

In [ ]:
df.columns = (
    df.columns
    .str.lower()
    .str.strip()
    .str.replace(' ', '_')
)

In [ ]:
df['departamento'] = (
    df['departamento']
    .str.upper()
    .str.strip()
)

In [ ]:
df['departamento'].unique()

# 2.2 Clasificación y agrupación de estratos socioeconómicos

In [ ]:
def clasificar_estrato(estrato):

    if estrato in [1, 2]:
        return 'Bajo'

    elif estrato in [3, 4]:
        return 'Medio'

    elif estrato in [5, 6]:
        return 'Alto'

    else:
        return 'Sin clasificar'

In [ ]:
df['categoria_estrato'] = df['estrato'].apply(clasificar_estrato)

In [ ]:
df[['estrato', 'categoria_estrato']].head()

In [ ]:
df['categoria_estrato'].value_counts()

# 2.3 Construcción de indicadores descriptivos

In [ ]:
indicadores = df.groupby('departamento').agg({
    'estrato': [
        'count',
        'mean',
        'min',
        'max'
    ]
})

indicadores

In [ ]:
porcentajes = pd.crosstab(
    df['departamento'],
    df['categoria_estrato'],
    normalize='index'
) * 100

porcentajes

In [ ]:
indicadores.to_csv('../exports/indicadores_departamento.csv')

# 2.4 Elaboración de tablas y gráficos comparativos

In [ ]:
plt.figure(figsize=(12,6))

sns.countplot(
    data=df,
    x='estrato',
    hue='departamento'
)

plt.title('Distribución de estratos por departamento')
plt.xlabel('Estrato')
plt.ylabel('Cantidad')

plt.show()

In [ ]:
plt.figure(figsize=(10,6))

sns.boxplot(
    data=df,
    x='departamento',
    y='estrato'
)

plt.title('Comparación territorial de estratos')

plt.show()

In [ ]:
tabla = pd.crosstab(
    df['departamento'],
    df['estrato']
)

plt.figure(figsize=(10,5))

sns.heatmap(
    tabla,
    annot=True,
    fmt='d'
)

plt.title('Mapa de calor de estratos')

plt.show()

In [ ]:
df['categoria_estrato'].value_counts().plot.pie(
    autopct='%1.1f%%'
)

plt.title('Distribución general de categorías')

plt.ylabel('')

plt.show()

# 2.5 Modelo descriptivo de comparación territorial entre departamentos

In [ ]:
comparacion = pd.crosstab(
    df['departamento'],
    df['categoria_estrato'],
    normalize='index'
) * 100

comparacion

In [ ]:
comparacion.plot(
    kind='bar',
    stacked=True,
    figsize=(12,6)
)

plt.title('Comparación territorial de estratos')

plt.ylabel('Porcentaje')

plt.show()

### Interpretación de resultados

Los resultados muestran diferencias en la distribución socioeconómica entre los departamentos analizados. Se evidencia una mayor concentración de estratos bajos en algunos territorios, mientras que otros presentan una distribución más equilibrada.

# 2.6 Implementación de dashboards

In [ ]:
fig = px.bar(
    df,
    x='estrato',
    color='departamento',
    barmode='group',
    title='Distribución de estratos por departamento'
)

fig.show()

In [ ]:
df.to_csv(
    '../data/processed/estratificacion_limpia.csv',
    index=False
)

# Conclusiones

- Se realizó exitosamente el proceso de limpieza y depuración de los datos.
- Se clasificaron los estratos socioeconómicos en categorías comparables.
- Se construyeron indicadores descriptivos para analizar la distribución territorial.
- Los gráficos permitieron identificar diferencias entre Cauca, Antioquia y Cundinamarca.
- Los dashboards facilitaron la visualización e interpretación de la información.